In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [5]:
len(parsed_docs)

95

In [6]:
from minsearch import AppendableIndex

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [10]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [11]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [12]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [14]:
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [18]:
print(response.output_text)

The CONTEXT does not provide specific instructions on how to create a dashboard in Evidently. It mentions the comprehensive feature set of the Evidently Platform, including rich dashboards, but does not detail the steps to create one.


Making It Agentic

In [ ]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: if you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [20]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.usage.input_tokens

63

In [21]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

110

In [22]:
tool_call = response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_3Ie7kJbSRJIEQihm9nMoTYRO', name='search', type='function_call', id='fc_05f152c58da087ae006ac536aeee8887d1bd67fb34261af251', async_=None, caller=None, namespace=None, status='completed')

In [23]:
messages.append(tool_call)

In [24]:
tool_call.arguments

'{"query":"create dashboard in Evidently"}'

In [25]:
arguments = json.loads(tool_call.arguments)
arguments

{'query': 'create dashboard in Evidently'}

In [26]:
search_results = search(query='create dashboard in Evidently')

In [27]:
search_results

[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

In [28]:
search_results = search(**arguments)
search_results[:1]

[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

## Continuing the Conversation


In [29]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [30]:
messages.append(call_output)
messages


[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: if you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_3Ie7kJbSRJIEQihm9nMoTYRO', name='search', type='function_call', id='fc_05f152c58da087ae006ac536aeee8887d1bd67fb34261af251', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_3Ie7kJbSRJIEQihm9nMoTYRO',
  'output': '[{"start": 0, "content": "<Check>\\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\n## What is a Dashboard?\\n\\nA Dashboard provides a clear view of your AI application performance. You can use it:\\n\\n- to track evaluation

## Getting the Final Response

In [37]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool]
    
)

In [38]:
import textwrap

import textwrap

def beautify_text(text, width=90):
    lines = text.strip().splitlines()
    wrapped_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            wrapped_lines.append("")
            continue

        wrapped_lines.append(
            textwrap.fill(
                line,
                width=width,
                subsequent_indent="   "
            )
        )

    return "\n".join(wrapped_lines)

print(beautify_text(response.output_text))

To create a dashboard in Evidently, follow these steps:

1. **Create a Project**: Ensure you have a project set up. Each project will have its own
   dashboard.

2. **Run an Evaluation**: Before you can create a dashboard, you need to run an evaluation
   and save at least one report to the project. This report will serve as your data
   source.

3. **Access the Dashboard**: Each project has a dashboard that starts empty. You can start
   populating it with panels.

4. **Add Panels**: You can add panels to display metrics. Panels can be created in two
   ways:
- **Using the UI** (available in Evidently Cloud and Enterprise).
- **Using the Python API**:
- If using the API, you'll first need to connect to your project:
```python
from evidently.sdk.models import PanelMetric
from evidently.sdk.panels import DashboardPanelPlot

# Example code to add a panel
project.dashboard.add_panel(
DashboardPanelPlot(
title="Panel Title",
size="full",
values=[PanelMetric(legend="Metric Legend", metric="

## Structured Output

In [39]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [40]:
response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

3909

In [41]:
rag_response = response.output_parsed
print(rag_response.answer)

To create a dashboard in Evidently, follow these steps:

### 1. **Create a Project**  
To start, ensure you have created a project in Evidently. A dashboard is linked to a specific project.

### 2. **Run an Evaluation**  
You need to run at least one evaluation and **save a Report** to your Project. The dashboard will be populated based on data from the Reports you generate.

### 3. **Access the Dashboard**  
You can access your Project’s dashboard, which will initially be empty.

### 4. **Add Panels**  
You can add **Panels** to visualize specific metrics. You have two options to do this:
- **Using Python API**:
  ```python
  from evidently.sdk.models import PanelMetric
  from evidently.sdk.panels import DashboardPanelPlot

  project.dashboard.add_panel(
      DashboardPanelPlot(
          title="Panel Title",
          values=[PanelMetric(legend="Metric Legend", metric="Metric Name")],
          plot_params={"plot_type": "line"},
      ),
      tab="Your Tab Name"
  )
  ```
- **Throu

## Agentic RAG vs Traditional RAG

The key difference:

Traditional RAG: - Always searches - Always includes results in prompt - Fixed flow: search, retrieve, generate - Can't handle typos or reformulate queries

Agentic RAG: - LLM chooses whether to search - LLM chooses the search query - Can fix typos and reformulate queries - Can use multiple tools in sequence - Continues until it has enough information

This makes agentic RAG more flexible. The LLM can: - Skip search if it already knows the answer - Reformulate the query if needed - Use multiple tools together - Reason about when it has enough information

The tradeoff is cost - each round trip adds tokens to the conversation. You saw the progression: 63 → 110 → 4075 → 4299.

